# 🤖 Notebook 06: 中国现货电价预测 (LEAR)

## 学习目标
1. 理解 LEAR (Lasso Estimated AutoRegressive) 的核心思想
2. 学会用电价滞后特征 + Lasso 正则化做日前预测
3. 理解 L1 正则化如何自动筛选重要特征
4. 对比 LEAR vs 持续法的预测效果
5. 读懂 Lasso 系数（正/负/零的含义）

## LEAR 原理

### 类比: 精明交易员的学习过程

假设你是一名新手电力交易员，第一周什么也不懂：

**第1天:** "昨天电价 300 元，今天大概也差不多" → 持续法

**第2天:** "哦，周末电价会低一些。昨天周五 350，但今天是周六" → 开始学模式

**第3天:** "我发现下午 4-6 点光伏出力下降时电价会跳升" → 更多模式

**第30天:** "我总结了 14 个规律，但有些规律其实没什么用" → 过度自信

**LEAR 的做法:**
- 先给你所有可能规律（14 个特征）
- 用 L1 正则化自动过滤掉不重要的规律（系数设为 0）
- 只保留真正有预测能力的几个特征

### Lasso (L1 正则化) 原理

**普通线性回归:**
$$\text{Minimize: } \frac{1}{n}\sum(y_i - \hat{y}_i)^2$$

**Lasso 回归:**
$$\text{Minimize: } \frac{1}{n}\sum(y_i - \hat{y}_i)^2 + \alpha \sum|\beta_i|$$

L1 惩罚项 $\alpha \sum|\beta_i|$ 会迫使不重要特征的系数趋近于零。

| 方法 | 原理 | 特征选择 | 可解释性 |
|------|------|---------|---------|
| **持续法** | 昨天=今天 | ❌ 无 | ✅ 最高 |
| **LEAR (Lasso)** | 线性 + L1 正则化 | ✅ 自动筛选 | ✅ 系数=边际影响 |
| **XGBoost** | 100棵树逐步修正 | ✅ 内置重要性 | ❌ 黑盒 |

为什么 LEAR 在电价预测中很有效？
因为电价有很强的**自相关性**（今天价格 → 明天价格），
线性模型配合滞后特征就能捕捉大部分模式。

In [ ]:
import sys, os
sys.path.insert(0, os.path.join('..', '..'))

from ellectric.pipeline.price_loader import PriceDataLoader, create_price_loader
from ellectric.pipeline.price_forecaster import LEARForecaster

import pandas as pd
import numpy as np
from sklearn.metrics import r2_score
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio

pio.renderers.default = 'notebook'

---
## 加载中国电价数据

数据来源: ZionLuo/Electricity-Price-Forecasting 仓库中的 `price data.xlsx`。
共 7 列: 日前价格 | 实时价格 | 统调负荷 | 风电 | 光伏 | 省间联络线。
约 2000 条小时级记录，覆盖中国某省份短期现货市场。

In [ ]:
# ── 加载电价数据 ──────────────────────────────────
try:
    loader = create_price_loader()
    df = loader.load_data()
except FileNotFoundError as e:
    print(e)
    print("\n⚠ 请先下载 price data.xlsx 到 data/ 目录:")
    print("  https://github.com/ZionLuo/Electricity-Price-Forecasting")
    print("将文件重命名为 price_data.xlsx 后重新运行此 cell")
    raise

# ── 数据量检查 ────────────────────────────────────
if len(df) < 168:
    raise ValueError(
        f"电价数据不足 168 行 (当前 {len(df)} 行)，"
        f"无法生成滞后 168h 特征"
    )

# ── 目标列验证 ────────────────────────────────────
target_col = 'price_da'
if target_col not in df.columns:
    print(f"目标列 '{target_col}' 不存在。可选列: {list(df.columns)}")
    raise AssertionError(f"目标列 {target_col} 不存在于数据中")

# ── 时间连续性检查 ────────────────────────────────
time_diffs = df['timestamp'].diff()
gap_count = (time_diffs > pd.Timedelta(hours=2)).sum()
if gap_count > 0:
    print(f"⚠ 数据存在 {gap_count} 处时间不连续 (gap > 2h)")

# ── 负价格检查（中国现货可能出现负电价） ─────────
n_neg = (df[target_col] < 0).sum()
print(f"price_da: 含 {n_neg} 个负值，"
      f"范围 [{df[target_col].min():.0f}, {df[target_col].max():.0f}]")
if n_neg > 0:
    print("  → 负电价是正常市场现象，LEAR 模型可处理负值")

# ── 统计摘要 ──────────────────────────────────────
print(f"\n数据形状: {df.shape}")
print(f"列列表: {list(df.columns)}")
print(f"时间范围: {df['timestamp'].min()} ~ {df['timestamp'].max()}")
print(f"时间频率: ~{pd.infer_freq(df['timestamp'])}")
print(f"\n{df.describe().round(2)}")
df.head()

---
## 特征工程

LEAR 的特征分为 3 个层级 (Tier 1-3)，逐步增加复杂度：

| 层级 | 特征数量 | 包含 |
|------|---------|------|
| **Tier 1** | 6 | 日历特征 + 价格滞后 (昨日/7天前) |
| **Tier 2** | 11 (+5) | 相关变量滞后 (负荷/风电/光伏) + 滚动统计 |
| **Tier 3** | 14 (+3) | 循环时间编码 + 价格趋势 |

每个层级都建立在上一级基础上。

In [ ]:
# ── 特征工程 — Tier 1: 日历 + 价格滞后 ─────────
forecaster = LEARForecaster(alpha=0.01)
df = forecaster.add_price_features(df, 'tier1')

feature_desc_t1 = {
    'hour': '小时 (0-23)',
    'day_of_week': '星期几 (0=周一, 6=周日)',
    'month': '月份 (1-12)',
    'is_weekend': '是否周末 (0=工作日, 1=周末)',
    'lag_24h_price': '昨日同时刻电价 (元/MWh)',
    'lag_168h_price': '7天前同时刻电价 (元/MWh)',
}

tier1_cols = forecaster.get_feature_columns('tier1')
print(f"Tier 1 特征 ({len(tier1_cols)} 个):")
for c in tier1_cols:
    desc = feature_desc_t1.get(c, '')
    print(f"  • {c:25s} {desc}")
print(f"\n数据维度: {df.shape}")

In [ ]:
# ── 特征工程 — Tier 2: 相关变量滞后 + 滚动统计 ─
df = forecaster.add_price_features(df, 'tier2')

tier2_all = forecaster.get_feature_columns('tier2')
tier2_new = [c for c in tier2_all if c not in tier1_cols]

feature_desc_t2 = {
    'lag_24h_load': '昨日同时刻统调负荷 (MW)',
    'lag_24h_wind': '昨日同时刻风电出力 (MW)',
    'lag_24h_solar': '昨日同时刻光伏出力 (MW)',
    'rolling_mean_24h_price': '过去24h电价均值 (元/MWh)',
    'rolling_std_24h_price': '过去24h电价标准差',
}

print(f"Tier 2 新增特征 ({len(tier2_new)} 个):")
for c in tier2_new:
    desc = feature_desc_t2.get(c, '')
    print(f"  • {c:30s} {desc}")
print(f"\n累计特征: {len(tier2_all)} 个")
print(f"数据维度: {df.shape}")

In [ ]:
# ── 特征工程 — Tier 3: 循环编码 + 价格趋势 ────
df = forecaster.add_price_features(df, 'tier3')

tier3_all = forecaster.get_feature_columns('tier3')
tier3_new = [c for c in tier3_all if c not in tier2_all]

feature_desc_t3 = {
    'hour_sin': '小时正弦编码 (sin(2π·h/24))',
    'hour_cos': '小时余弦编码 (cos(2π·h/24))',
    'price_trend_7d': '7天价格移动平均 (元/MWh)',
}

print(f"Tier 3 新增特征 ({len(tier3_new)} 个):")
for c in tier3_new:
    desc = feature_desc_t3.get(c, '')
    print(f"  • {c:25s} {desc}")
print(f"\n累计特征: {len(tier3_all)} 个")
print(f"数据维度: {df.shape}")
print(f"\n全部特征: {tier3_all}")

---
## LEAR 模型训练与评估

使用 TimeSeriesSplit (n_splits=5, gap=24h) 确保：
- 训练数据永远在测试数据之前 → 防止 look-ahead bias
- gap=24h 防止邻近时间数据泄露

### 评估指标

**MAE (平均绝对误差):**
$$MAE = \frac{1}{n}\sum|y_i - \hat{y}_i|$$
"平均来说，预测偏差 ± X 元/MWh"

**RMSE (均方根误差):**
$$RMSE = \sqrt{\frac{1}{n}\sum(y_i - \hat{y}_i)^2}$$
对大误差惩罚更重，RMSE > MAE 说明存在极端误差。

**MAPE (平均绝对百分比误差):**
$$MAPE = \frac{100}{n}\sum\left|\frac{y_i - \hat{y}_i}{y_i}\right|$$
MAPE=10% 表示偏差约占实际价格的 10%。

**R² (决定系数):**
$$R² = 1 - \frac{\sum(y_i - \hat{y}_i)^2}{\sum(y_i - \bar{y})^2}$$
R²=1 完美; R²=0 不比均值好; R²<0 比瞎猜还差。

### 对比基准: 持续法 (Persistence)

持续法假设 **今天电价 = 昨天同时刻电价**。
如果 LEAR 不如持续法，说明特征工程或参数配置有问题。

In [ ]:
# ── LEAR 模型训练 ────────────────────────────────
n_splits = 5
gap = 24

# 边界检查: n_splits * gap 是否占用过多训练数据
train_ratio = n_splits / (n_splits + 1)
train_est = int(len(df) * train_ratio)
gap_ratio = (n_splits * gap) / train_est
if gap_ratio > 0.3:
    print(f"⚠ 训练集 {n_splits} 折 × gap {gap}h 可能占用过多训练数据 "
          f"(gap 占训练集 {gap_ratio:.0%})")

result = forecaster.train_evaluate(df, tier='tier3',
                                    n_splits=n_splits, gap=gap)

metrics = result['metrics']
predictions = result['predictions']
actuals = result['actuals']

# ── 计算补充指标 (MAPE, R²) ──────────────────────
mape = np.mean(np.abs((actuals - predictions)
                       / np.where(actuals != 0, actuals, 1e-10))) * 100
r2 = r2_score(actuals, predictions)

print(f"═══ LEAR (Tier 3) 评估结果 ═══")
print(f"MAE:  {metrics['mae']:.2f} 元/MWh")
print(f"RMSE: {metrics['rmse']:.2f} 元/MWh")
print(f"MAPE: {mape:.1f}%")
print(f"R²:   {r2:.3f}")

# ── 特征系数统计 ──────────────────────────────────
coefs = result['model'].coef_
n_zero = sum(1 for c in coefs if abs(c) < 1e-10)
print(f"\nLasso 系数: {len(coefs)} 个特征中 {n_zero} 个被压缩为零 "
      f"({n_zero/len(coefs)*100:.0f}% 稀疏率)")

# ── 对比持续法 ────────────────────────────────────
persist_vals = df['price_da'].shift(24)
valid_mask = persist_vals.notna()
persist_mae = (persist_vals[valid_mask]
               - df.loc[valid_mask, 'price_da']).abs().mean()

print(f"\n═══ 模型对比 ═══")
print(f"持续法 MAE ({valid_mask.sum()}/{len(df)} 有效样本): "
      f"{persist_mae:.2f} 元/MWh")
print(f"LEAR  MAE:                                 "
      f"{metrics['mae']:.2f} 元/MWh")

improvement = (persist_mae - metrics['mae']) / persist_mae * 100
if improvement > 0:
    print(f"\n✅ LEAR 提升了 {improvement:.1f}% — 比持续法更好!")
else:
    print(f"\n⚠ LEAR 没比持续法好 (差 {abs(improvement):.1f}%)，"
          f"可能特征工程或 alpha 参数有问题")

---
## 预测结果可视化

两个图：
1. **实际 vs 预测叠加图** — 直观对比时间序列
2. **预测误差分布图** — 查看误差是否集中在 0 附近
3. **特征系数图** — 哪些特征对预测影响最大

In [ ]:
# ── LEAR 预测 vs 实际 ────────────────────────────
fig_forecast = forecaster.plot_price_forecast(df, predictions)
fig_forecast.show()

In [ ]:
# ── Lasso 特征系数 ───────────────────────────────
fig_coef = forecaster.plot_coefficients(top_n=14)
fig_coef.show()

# 打印系数解读
coef_df = pd.DataFrame({
    '特征': forecaster.get_feature_columns('tier3'),
    '系数': result['model'].coef_,
    '|系数|': np.abs(result['model'].coef_),
}).sort_values('|系数|', ascending=False)
print("\n特征系数排名 (按 |系数| 降序):")
for _, row in coef_df.iterrows():
    bar = '█' * int(abs(row['系数']) / coef_df['|系数|'].max() * 20)
    direction = '↑ 价格推高' if row['系数'] > 0 else '↓ 价格压低'
    if abs(row['系数']) < 1e-10:
        direction = '· 被L1剔除'
    print(f"  {row['特征']:25s} {bar} ({row['系数']:+.4f}) {direction}")

---
## 📝 学习笔记

- LEAR = Lasso + 滞后特征，是电价预测的经典基线方法
- L1 正则化自动筛选重要特征 — 不重要的系数会被压缩到零
- 正系数 = 特征值 ↑ → 价格 ↑；负系数 = 特征值 ↑ → 价格 ↓
- TimeSeriesSplit(gap=24) 防止 look-ahead bias
- 持续法是最低标准 — 任何模型都应该超过它

## 🤔 思考题

1. 哪些特征的系数被 L1 压缩为零了？为什么这些特征不相关？
2. 如果持续法比 LEAR 还好，可能是哪些原因？请列出至少 3 种可能性。
3. 对比 Phase 1 的 XGBoost 模型：LEAR 的可解释性更强还是更弱？在什么场景下你会选择 LEAR 而非 XGBoost？
4. 循环时间编码 (hour_sin/hour_cos) 相比直接用 hour (0-23) 的优缺点是什么？
5. 如果给你更多数据（3 年的电价数据而非 3 个月），你会添加哪些额外的特征？